# Preparation Des Donnees LLM Support

Ce notebook prepare les jeux de donnees d'entrainement et de test a partir du CSV brut des tickets de support.


## Installer Les Dependances

Installez les dependances du projet dans l'environnement actif du notebook avant d'executer le reste du notebook.


In [1]:
%pip install -r requirements.txt


  Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl (73.6 MB)
     |████████████████████████████████| 12.0 MB 11.2 MB/s eta 0:00:01
     |████████████████████████████████| 515 kB 29.3 MB/s eta 0:00:01
     |████████████████████████████████| 504 kB 48.3 MB/s eta 0:00:01
     |████████████████████████████████| 423 kB 48.0 MB/s eta 0:00:01
     |████████████████████████████████| 374 kB 23.6 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 22.9 MB/s eta 0:00:01
  Using cached filelock-3.19.1-py3-none-any.whl (15 kB)
     |████████████████████████████████| 200 kB 53.6 MB/s eta 0:00:01
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
     |████████████████████████████████| 447 kB 26.7 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 28.9 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 14.1 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 34.4 MB/s eta 0:00:01
     |

## Imports

Nous avons seulement besoin de pandas pour manipuler les donnees et de scikit-learn pour la separation stratifiee.


In [2]:
import json
import pandas as pd
from support_classification.prompts import build_prompt
from support_classification.paths import LABELS_PATH, RAW_CSV_PATH, TRAIN_DATASET_PATH, TEST_DATASET_PATH, RAW_DIR, PROCESSED_DIR
from sklearn.model_selection import train_test_split

# Rend les apercus de DataFrame plus lisibles dans le notebook.
pd.set_option('display.max_colwidth', 120)


## Organiser Les Dossiers De Donnees

Nous conservons le CSV brut dans `data/raw/` et ecrivons les sorties preparees dans `data/processed/`.


In [3]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Raw CSV path:', RAW_CSV_PATH)
print('Processed dir:', PROCESSED_DIR)


Raw CSV path: /Users/jadedupont/projects/blentAI/llm/projets-hands-on/blentai-llm-support-classification/data/raw/helpdesk_customer_tickets.csv
Processed dir: /Users/jadedupont/projects/blentAI/llm/projets-hands-on/blentai-llm-support-classification/data/processed


## Charger Le Jeu De Donnees

Nous chargeons le CSV depuis `data/raw/` et inspectons sa structure avant de ne garder que les colonnes autorisees.


In [4]:
df = pd.read_csv(RAW_CSV_PATH)

# Apercu rapide du fichier brut.
df.head()


,id,subject,body,answer,type,queue,priority,language,business_type,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,tag_9
0,36,Anfrage zu den Spezifikationen und Anpassungsoptionen des MacBook Air M1,"Sehr geehrtes Support-Team des Tech Online Stores,\n\nich interessiere mich für den Kauf eines MacBook Air M1 und hä...","Sehr geehrter <name>,\n\nvielen Dank für Ihr Interesse. Das MacBook Air M1 verfügt über ein 13,3"" Retina-Display, ei...",Request,Customer Service,medium,de,Tech Online Store,Product Support,Sales Inquiry,Technical Guidance,General Inquiry,NaN,NaN,NaN,NaN,NaN
1,39,Déconnexions fréquentes et plantages,Le client signale des déconnexions fréquentes et des plantages lors des réunions vidéo utilisant Zoom 5.11.0. Veuill...,"Nous allons enquêter sur le problème avec Zoom 5.11.0. En attendant, assurez-vous que vous utilisez la dernière vers...",Incident,Product Support,high,fr,Software Development Company,Technical Support,Software Bug,Service Disruption,System Crash,Problem Resolution,Performance Tuning,NaN,NaN,NaN
2,243,Problema de sonido Dell XPS,"Problema con el sonido, manejando como devolución. Dentro de 30 días.",Gracias por su correo electrónico. Procesaremos su devolución para el Dell XPS. Espere más instrucciones pronto.,Problem,Returns and Exchanges,medium,es,Tech Online Store,Returns and Exchanges,Product Support,Customer Service,Refund Request,NaN,NaN,NaN,NaN,NaN
3,381,Assistance requise pour la configuration du tableau Scrum,"Cher support client,\n\nNotre client, <name>, a besoin d'aide pour configurer un tableau Scrum dans Jira Software 8....","Cher <name>,\n\nMerci de nous avoir contactés. Pour configurer un tableau Scrum dans Jira Software 8.20, allez dans ...",Request,Product Support,medium,fr,Software Development Company,Technical Support,Product Support,General Inquiry,Problem Resolution,Training Request,NaN,NaN,NaN,NaN
4,663,Urgente: Assistência Imediata Necessária para Falha na Aplicação de Folha de Pagamento,"Caro Suporte ao Cliente da Firma de Consultoria de TI, Nosso RH relata que nossa aplicação de folha de pagamento pri...","Caro Cliente,\n\nRecebemos sua solicitação urgente sobre a falha na aplicação de folha de pagamento. Nossa equipe es...",Incident,Human Resources,medium,pt,IT Consulting Firm,Urgent Issue,Payroll Issue,Technical Support,Service Disruption,Problem Resolution,Account Assistance,NaN,NaN,NaN


## Ne Garder Que Les Colonnes Autorisees

La specification autorise explicitement uniquement ces variables comme entrees du modele et cible : `subject`, `body`, `queue`, `language` et `business_type`.


In [5]:
cols = ['subject', 'body', 'queue', 'language', 'business_type']
df = df[cols].copy()

# Supprime les lignes incompletes afin que chaque exemple possede les champs necessaires pour construire un prompt.
df = df.dropna(subset=cols).reset_index(drop=True)

df.info()

# Construit une seule fois l'ensemble ferme des labels de queue afin que la generation
# des prompts, l'entrainement et l'evaluation reposent tous sur la meme liste de reponses autorisees.
valid_labels = sorted(df['queue'].unique())
print('Valid labels:', valid_labels)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599 entries, 0 to 598
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   subject        599 non-null    object
 1   body           599 non-null    object
 2   queue          599 non-null    object
 3   language       599 non-null    object
 4   business_type  599 non-null    object
dtypes: object(5)
memory usage: 23.5+ KB
Valid labels: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']


## Separation Stratifiee Train/Test

Nous separons le jeu de donnees en preservant la distribution des classes de `queue` dans les deux sous-ensembles.


In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    # Preserve les proportions initiales des classes de `queue` dans le train et le test.
    stratify=df['queue']
)

print('Train size:', len(train_df))
print('Test size:', len(test_df))

# Affiche la distribution des classes en pourcentage.
print('\nTrain class distribution:')
print((train_df['queue'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

print('\nTest class distribution:')
print((test_df['queue'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')


Train size: 479
Test size: 120

Train class distribution:
queue
Billing and Payments                7.72 %
Customer Service                    14.2 %
General Inquiry                     0.84 %
Human Resources                     2.51 %
IT Support                         12.94 %
Product Support                    15.45 %
Returns and Exchanges               6.68 %
Sales and Pre-Sales                 2.09 %
Service Outages and Maintenance     2.51 %
Technical Support                  35.07 %
Name: proportion, dtype: object

Test class distribution:
queue
Billing and Payments                 7.5 %
Customer Service                   14.17 %
General Inquiry                     0.83 %
Human Resources                      2.5 %
IT Support                          12.5 %
Product Support                    15.83 %
Returns and Exchanges               6.67 %
Sales and Pre-Sales                  2.5 %
Service Outages and Maintenance      2.5 %
Technical Support                   35.0 %
Name: propor

## Construire Les Jeux De Donnees Train Et Test

Nous creons une colonne `prompt` pour l'entree et une colonne `answer` pour le label attendu.


In [7]:
train_df = train_df.copy()
test_df = test_df.copy()

# Cree des copies de travail independantes afin de pouvoir ajouter de nouvelles colonnes sans effets de bord.
# axis=1 signifie que la fonction est appliquee ligne par ligne.
train_df['prompt'] = train_df.apply(
    lambda row: build_prompt(
        subject=row['subject'],
        body=row['body'],
        language=row['language'],
        business_type=row['business_type'],
        valid_labels=valid_labels,
    ),
    axis=1,
)
train_df['answer'] = train_df['queue']

test_df['prompt'] = test_df.apply(
    lambda row: build_prompt(
        subject=row['subject'],
        body=row['body'],
        language=row['language'],
        business_type=row['business_type'],
        valid_labels=valid_labels,
    ),
    axis=1,
)
test_df['answer'] = test_df['queue']

# Ne conserve que le prompt et le label attendu pour creer un jeu de donnees d'entrainement propre.
train_dataset = train_df[['prompt', 'answer']].copy()
test_dataset = test_df[['prompt', 'answer']].copy()


## Inspecter Les Exemples Prepares

Un apercu rapide permet de verifier que le format du prompt est coherent et lisible.


In [8]:
display(train_dataset.head(2))


,prompt,answer
120,You are a support ticket classification assistant.\nChoose exactly one support ticket category from the allowed labe...,Product Support
426,You are a support ticket classification assistant.\nChoose exactly one support ticket category from the allowed labe...,Returns and Exchanges


In [9]:
display(test_dataset.head(2))


,prompt,answer
203,You are a support ticket classification assistant.\nChoose exactly one support ticket category from the allowed labe...,Technical Support
125,You are a support ticket classification assistant.\nChoose exactly one support ticket category from the allowed labe...,Technical Support


## Exporter Les Jeux De Donnees Traites

Les fichiers CSV prepares et le fichier partage `labels.json` sont ecrits dans `data/processed/` pour etre reutilises dans le notebook d'entrainement.


In [10]:
# Sauvegarde les jeux de donnees prepares et la liste des labels autorises pour une reutilisation ulterieure.
train_dataset.to_csv(TRAIN_DATASET_PATH, index=False)
test_dataset.to_csv(TEST_DATASET_PATH, index=False)
LABELS_PATH.write_text(json.dumps({'labels': valid_labels}, indent=2), encoding='utf-8')

print('Saved train dataset to:', TRAIN_DATASET_PATH)
print('Saved test dataset to:', TEST_DATASET_PATH)
print('Saved labels to:', LABELS_PATH)


Saved train dataset to: /Users/jadedupont/projects/blentAI/llm/projets-hands-on/blentai-llm-support-classification/data/processed/train_dataset.csv
Saved test dataset to: /Users/jadedupont/projects/blentAI/llm/projets-hands-on/blentai-llm-support-classification/data/processed/test_dataset.csv
Saved labels to: /Users/jadedupont/projects/blentAI/llm/projets-hands-on/blentai-llm-support-classification/data/processed/labels.json
